In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# ==========================================
# Load
# ==========================================

df = pd.read_csv(
    "data/market_data_final_features.csv",
    parse_dates=["Date"],
    index_col="Date"
)

df = df.dropna(
    subset=["HMM_Regime_Risk"]
).copy()

TARGET = "Crash_Risk"

# ==========================================
# Leakage / unused columns
# ==========================================

exclude = [
    TARGET,
    "Future_Min_SP500",
    "Future_Drawdown_10D",
    "Market_Regime",
    "HMM_State",
    "PELT_Recent_Change",
    "SP500",
    "NASDAQ",
    "GOLD",
    "OIL",
    "SP500_MA20",
    "SP500_MA50"
]

all_features = [
    c for c in df.columns
    if c not in exclude
]

# ==========================================
# Feature Groups
# ==========================================

tda_features = [
    c for c in all_features
    if c.startswith("TDA_")
]

pelt_features = [
    c for c in all_features
    if c.startswith("PELT_")
]

hmm_features = [
    "HMM_Regime_Risk"
]

market_features = [
    c for c in all_features
    if c not in (
        tda_features
        + pelt_features
        + hmm_features
    )
]

feature_sets = {

    "Market Only":
        market_features,

    "Market + TDA":
        market_features
        + tda_features,

    "Market + TDA + PELT":
        market_features
        + tda_features
        + pelt_features,

    "Full Model":
        market_features
        + tda_features
        + pelt_features
        + hmm_features
}

# ==========================================
# Same chronological split
# ==========================================

split = int(
    len(df) * 0.80
)

y = df[TARGET]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

weight = negative / positive

print(
    "Train:",
    len(y_train),
    "Test:",
    len(y_test)
)

print(
    "Test crash rate:",
    round(y_test.mean(), 4)
)

print(
    "Random PR baseline:",
    round(y_test.mean(), 4)
)

# ==========================================
# Train Models
# ==========================================

results = []

for name, features in feature_sets.items():

    print(
        f"\nTraining: {name}"
    )

    print(
        "Features:",
        len(features)
    )

    X = df[features]

    X_train = X.iloc[:split]
    X_test = X.iloc[split:]

    model = CatBoostClassifier(
        iterations=700,
        depth=6,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="AUC",
        scale_pos_weight=weight,
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(
            X_test,
            y_test
        ),
        early_stopping_rounds=100,
        verbose=False
    )

    prob = model.predict_proba(
        X_test
    )[:, 1]

    roc = roc_auc_score(
        y_test,
        prob
    )

    pr = average_precision_score(
        y_test,
        prob
    )

    pred = (
        prob >= 0.50
    ).astype(int)

    tp = (
        (pred == 1)
        & (y_test.values == 1)
    ).sum()

    fp = (
        (pred == 1)
        & (y_test.values == 0)
    ).sum()

    fn = (
        (pred == 0)
        & (y_test.values == 1)
    ).sum()

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0
    )

    results.append({
        "Model": name,
        "Features": len(features),
        "ROC_AUC": roc,
        "PR_AUC": pr,
        "Precision_05": precision,
        "Recall_05": recall,
        "F1_05": f1
    })

# ==========================================
# Results
# ==========================================

results_df = pd.DataFrame(
    results
)

print(
    "\n========== ABLATION RESULTS =========="
)

print(
    results_df.to_string(
        index=False
    )
)

results_df.to_csv(
    "data/ablation_results.csv",
    index=False
)

print(
    "\nSaved: data/ablation_results.csv"
)

Train: 4738 Test: 1185
Test crash rate: 0.0346
Random PR baseline: 0.0346

Training: Market Only
Features: 20

Training: Market + TDA
Features: 29

Training: Market + TDA + PELT
Features: 35

Training: Full Model
Features: 36

========== ABLATION RESULTS ==========
              Model  Features  ROC_AUC   PR_AUC  Precision_05  Recall_05    F1_05
        Market Only        20 0.782918 0.096483      0.105991   0.560976 0.178295
       Market + TDA        29 0.836538 0.154652      0.173611   0.609756 0.270270
Market + TDA + PELT        35 0.820484 0.112740      0.136054   0.487805 0.212766
         Full Model        36 0.846793 0.130093      0.114894   0.658537 0.195652

Saved: data/ablation_results.csv
